# CyberShield: AI-Powered Cyberbullying Detection

## College Project Report

- Department: [Your Department]
- Institution: [Your University / College]
- Academic Term: [Semester, Year]
- Author(s): [Your Name(s)]
- Guide: [Faculty Guide Name]
- Submission Date: [Month Year]

## Abstract

CyberShield is a lightweight, production-ready web application that detects cyberbullying using Natural Language Processing (NLP) and machine learning. It combines a TF–IDF text feature extractor with multi-label classifiers to predict probabilities for toxicity categories such as toxic, severe_toxic, obscene, threat, insult, and identity_hate, and derives an overall bullying indicator. The Flask backend serves a clean web interface and REST endpoints, while a generative AI model (Google Gemini) synthesizes human-readable conclusions from model outputs. This report presents the problem context, motivation, literature background, system architecture, dataset, end-to-end ML pipeline (data cleaning, preprocessing, imbalance handling, feature extraction, model training, evaluation), application code with line-by-line explanations, and deployment setup via Docker and Procfile. Image placeholders are included where visual artifacts (architecture diagram, EDA charts, and UI screenshots) will be added later.

## Table of Contents

1. Introduction
2. Literature Review
3. System Overview and Architecture
4. Dataset Description
5. Exploratory Data Analysis (EDA)
6. Data Preprocessing and Feature Engineering
7. Handling Class Imbalance
8. Model Development and Evaluation
9. Backend Implementation (Flask)
10. Frontend Pages and User Flow
11. Deployment: Docker and Procfile
12. Security, Ethics, and Privacy
13. Limitations and Future Work
14. Conclusion
15. References
16. Appendix: Full Code Listings and Artifacts

## 1. Introduction

### 1.1 Problem Statement
Cyberbullying is pervasive across social platforms and digital communication channels. Manual moderation alone cannot scale to the volume and speed of online interactions. Automated detection, especially probabilistic multi-label classification, provides an assistive layer for content safety teams and applications.

### 1.2 Objectives
- Build an end-to-end ML pipeline to classify toxic behaviors in text.
- Provide a simple, responsive web interface and API for real-time prediction.
- Generate human-readable summaries for users from model outputs.
- Package and deploy using standard DevOps artifacts.

### 1.3 Scope
CyberShield focuses on English-language short-form text. The scope includes model experimentation in Jupyter, a Flask backend for inference, and Docker-based deployment.

## 2. Literature Review

Prior work often uses TF–IDF or word embeddings (Word2Vec, GloVe, BERT family) with linear or tree-based classifiers for toxic comment classification. Multi-label setups (One-vs-Rest strategies) are common when each comment can exhibit multiple categories simultaneously. The Jigsaw Toxic Comment Classification dataset is widely referenced for benchmarking toxicity detection. While large transformer models offer state-of-the-art results, TF–IDF + linear models provide strong baselines with fast inference and low compute cost suitable for deployment on modest infrastructure.

## 3. System Overview and Architecture

### 3.1 High-Level Architecture
Insert architecture diagram here.

![CyberShield Architecture](static/Images/architecture.png)

Diagram description (to be visualized in the final image):
- UI Layer (Flask templates + static pages)
- REST Endpoints: /predict (model inference), /conclude (LLM summary)
- ML Inference: TF–IDF vectorizer + classifier (loaded from .pkl)
- Generative AI: Gemini 1.5 Flash for textual conclusions
- Config & Secrets: .env for API key (not committed)
- Containerization: Dockerfile (optional), Procfile for PaaS

### 3.2 Page and Request Flow
- Index Page (templates/index.html): Accepts user comment input and displays probabilities + conclusion.
- Static Pages (static/*.html): About, Awareness, Case Study, FAQ, Join Us for informational content.
- /predict: Accepts POST form data, cleans + vectorizes text, predicts per-label probabilities and a bullying flag.
- /conclude: Accepts JSON of the probabilities and bullying flag, prompts Gemini to generate a concise conclusion.

Sequence (to be shown in a sequence diagram image):
User -> UI -> /predict -> Model -> Probabilities -> /conclude -> Gemini -> Conclusion -> UI.

## 4. Dataset Description

The project uses a toxicity-labeled dataset with columns typical of the Jigsaw Toxic Comment Classification challenge: `comment_text` and binary labels `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate`.

Assumption note: The notebook loads `toxic_comments.csv` (local path). We assume this dataset mirrors Jigsaw's schema. Replace the path with your actual dataset location when reproducing.

Key properties to include (visualize later):
- Class distribution across labels (bar chart).
- Label correlations (heatmap).
- Text length distributions.
- Examples of toxic vs non-toxic comments (with sensitive text redacted).

## 5. Exploratory Data Analysis (EDA)

Produce the following visuals (add images later):
- Category Occurrence Bar Chart — counts per label.
- Correlation Heatmap — relationships among toxicity labels.
- Punctuation Analysis — boxplots and KDE of internal punctuation counts vs bullying flag.
- Word Clouds — per-label frequent words.

These guide feature engineering, class imbalance strategies, and model choice.

## 6. Data Preprocessing and Feature Engineering

### 6.1 Text Cleaning, Tokenization, Lemmatization
We lowercase, strip punctuation, URLs, and numeric tokens, tokenize, remove stopwords, and lemmatize to base forms.

### 6.2 TF–IDF Vectorization
We use `TfidfVectorizer` (1–2 grams, max_features=10,000) to transform text to numeric features suitable for linear classifiers.

In [ ]:
# Preprocessing utilities (from training notebook)
import re, string, nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Ensure required NLTK resources are available
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
# Note: 'punkt_tab' is not a standard NLTK package and may cause an error if invoked.

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def tokenize_and_lemmatize(text: str) -> str:
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

## 7. Handling Class Imbalance

The dataset is highly imbalanced (non-toxic >> toxic). Strategies used:
- Undersampling the majority (non-toxic) to match toxic counts for a balanced training subset.
- Targeted oversampling for rare unique label cases.
- Class weights in certain models to compensate for imbalance.

In [ ]:
# Imbalance handling (from training notebook excerpts)
import pandas as pd
from sklearn.utils import resample

# Example: undersample majority class
# non_toxic_data and toxic_data are derived by filtering Bullying == 0 or 1 respectively.
# non_toxic_downsampled = resample(non_toxic_data, replace=False, n_samples=len(toxic_data), random_state=42)
# new_df = pd.concat([non_toxic_downsampled, toxic_data]).sample(frac=1, random_state=42).reset_index(drop=True)

# Example: targeted oversampling of rare unique labels up to a max class size
# for label in ['toxic','severe_toxic','obscene','threat','insult','identity_hate']:
#     unique_class_data = new_df[(new_df[label]==1) & (new_df[['toxic','severe_toxic','obscene','threat','insult','identity_hate']].drop(label, axis=1)==0).all(axis=1)]
#     # replicate unique_class_data as needed and concat back to new_df

## 8. Model Development and Evaluation

### 8.1 Train/Test Split and Vectorization
We split the data (80/20) and fit a `TfidfVectorizer` on training text only to avoid leakage.

### 8.2 Classifiers Explored
- One-vs-Rest Logistic Regression (balanced class weights) — strong baseline.
- RandomForest (balanced_subsample).
- LinearSVC (balanced).
- Decision Tree (depth constraints).
- XGBoost (scale_pos_weight per label).
- KNN (distance-weighted).

### 8.3 Metrics
Per-label precision, recall, F1-score via `classification_report`. Word-clouds from LR coefficients help interpret top features per class.

In [ ]:
# Feature engineering and baseline model training (from training notebook)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
import numpy as np

# X = new_df['lemmatized_comment']
# y = new_df[['toxic','severe_toxic','obscene','threat','insult','identity_hate','Bullying']]
# vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
# X_tfidf = vectorizer.fit_transform(X)
# X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# model = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight='balanced'))
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)
# for i, label in enumerate(y.columns):
#     print(f'Classification report for {label}:')
#     print(classification_report(y_test.iloc[:, i], y_pred[:, i]))

In [ ]:
# Additional models (from training notebook, not necessarily selected for production)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.multiclass import OneVsRestClassifier
# from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier

# Examples (pseudo-execution for documentation):
# rf_model = OneVsRestClassifier(RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=42, n_jobs=-1))
# rf_model.fit(X_train, y_train)
# y_pred_rf = rf_model.predict(X_test)

# svm_model = OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=1000))
# svm_model.fit(X_train, y_train)
# y_pred_svm = svm_model.predict(X_test)

# dt_model = OneVsRestClassifier(DecisionTreeClassifier(class_weight='balanced', max_depth=10, random_state=42))
# dt_model.fit(X_train, y_train)
# y_pred_dt = dt_model.predict(X_test)

# xgb_classifiers = []
# for label in y.columns:
#     num_neg = (y[label] == 0).sum()
#     num_pos = (y[label] == 1).sum()
#     spw = num_neg / max(1, num_pos)
#     xgb = XGBClassifier(objective='binary:logistic', eval_metric='logloss', scale_pos_weight=spw, random_state=42)
#     xgb_classifiers.append(xgb)
# model_xgb = OneVsRestClassifier(estimator=xgb_classifiers[0])
# model_xgb.fit(X_train, y_train)

# knn_model = OneVsRestClassifier(KNeighborsClassifier(n_neighbors=5, weights='distance'))
# knn_model.fit(X_train, y_train)
# y_pred_knn = knn_model.predict(X_test)

### 8.4 Model Export for Inference
The deployed app expects `nmodel.pkl` and `nvectorizer.pkl`. After training, save them using joblib.

In [ ]:
# Save trained artifacts for deployment (to be executed after choosing the best model)
# import joblib
# joblib.dump(model, 'nmodel.pkl')
# joblib.dump(vectorizer, 'nvectorizer.pkl')

## 9. Backend Implementation (Flask)

This section explains `app.py` line-by-line: loading artifacts, preprocessing pipeline, routes (`/`, `/predict`, `/conclude`), and Gemini integration. Notes:
- NLTK downloads include `punkt_tab`, which is not standard and may raise an error — remove if unnecessary.
- `.env` is used for `GOOGLE_API_KEY` — keep the key secret and never commit it.
- `model.predict_proba` returns per-label probabilities; the app derives `is_bullying` using a 0.5 threshold on the seventh output.
- `/conclude` composes a natural-language summary using Gemini 1.5 Flash. Handle API errors gracefully.

In [ ]:
# File: app.py (full listing)
from flask import Flask, render_template, request, jsonify
import joblib
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import google.generativeai as genai
import os
from dotenv import load_dotenv

app = Flask(__name__)

# Load model and vectorizer
model = joblib.load('nmodel.pkl')
vectorizer = joblib.load('nvectorizer.pkl')

# Initialize NLTK components
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

load_dotenv()
os.getenv('GOOGLE_API_KEY')
genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))

def clean_text(text):
    """Cleans the input text."""
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text) 
    text = re.sub(r'http\S+', '', text)  
    text = re.sub(r'\w*\d\w*', '', text)  
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)  
    text = re.sub(r'\n', ' ', text)  
    text = re.sub(r'\s+', ' ', text)  
    return text

def preprocess_input(text):
    """Preprocess the input text for prediction."""
    text = clean_text(text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    processed_text = ' '.join(tokens)
    vectorized_text = vectorizer.transform([processed_text])
    return vectorized_text

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    comment = request.form['comment']
    
    # Preprocess and vectorize the input comment
    vectorized_text = preprocess_input(comment)
    prediction = model.predict_proba(vectorized_text)[0]
    
    result = {
        'toxic': prediction[0],
        'severe_toxic': prediction[1],
        'obscene': prediction[2],
        'threat': prediction[3],
        'insult': prediction[4],
        'identity_hate': prediction[5]
    }
    
    is_bullying = prediction[6]>=0.5

    return jsonify({'result': result, 'is_bullying': bool(is_bullying)})

@app.route('/conclude', methods=['POST'])
def conclude():
    print('called')
    # Extract predictions from request
    predictions = request.json.get('result')
    is_bullying = request.json.get('is_bullying')

    # Build a summary of the predictions
    summary = f"The comment is{' not' if not is_bullying else ''} indicative of cyberbullying. "
    summary += "Here are the detailed probabilities for each type of toxicity: "
    for label, score in predictions.items():
        summary += f"{label.capitalize()}: {score:.2f}. "
    # Generate conclusion using Google Gemini-Pro
    print(summary)
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(
        f"Given the following summary of toxicity predictions for a comment, provide a concise conclusion about the nature of the comment. The prediction is based on a machine learning model, and you should communicate the certainty of the prediction. Here are the detailed probabilities for each type of toxicity, also do not include the technical aspects like what models predict: {summary}"
    )
    print(response)
    try:
        candidates = response._result.candidates
        conclusion_text = candidates[0].content.parts[0].text
    except AttributeError as e:
        return jsonify({'error': 'Failed to extract conclusion text from the response: ' + str(e)}), 500
    except IndexError as e:
        return jsonify({'error': 'Index out of range when extracting conclusion text: ' + str(e)}), 500
    print(conclusion_text)
    return jsonify({'conclusion': conclusion_text})

if __name__ == '__main__':
    app.run(debug=True)

### 9.1 Line-by-Line Highlights
- `joblib.load('nmodel.pkl')` and `joblib.load('nvectorizer.pkl')` load trained artifacts.
- `clean_text` and `preprocess_input` must mirror training-time preprocessing to avoid data shift.
- `/predict` returns per-label probabilities and a boolean `is_bullying`.
- `/conclude` constructs a prompt and calls Gemini; robust error handling is included.
- Consider replacing internal `_result` access with stable response parsing if available in the SDK version.
- Remove `nltk.download('punkt_tab')` if it fails; it's not a standard package.

## 10. Frontend Pages and User Flow

The UI uses a Jinja2 template for the index page and static informational pages. Add screenshots later:
- Home/Input (index.html): text area for comment, submit button, result probabilities table, and AI-generated conclusion.
- About (static/about.html), Awareness, Case Study, FAQ, Join Us: informational content with shared `styles.css`.

Note: The contents of `templates/index.html` and `static/*.html` were not provided in the attachments. Place their full HTML here when finalizing the report.

In [ ]:
<!-- File: templates/index.html (content not provided in attachments) -->
<!-- Insert the full HTML template here when available. -->

In [ ]:
<!-- File: static/about.html (content not provided in attachments) -->

In [ ]:
<!-- File: static/awareness.html (content not provided in attachments) -->

In [ ]:
<!-- File: static/casestudy.html (content not provided in attachments) -->

In [ ]:
<!-- File: static/faq.html (content not provided in attachments) -->

In [ ]:
<!-- File: static/joinus.html (content not provided in attachments) -->

In [ ]:
/* File: static/styles.css (content not provided in attachments) */

## 11. Deployment: Docker and Procfile

Two deployment options:
- Docker container — self-contained runtime with Flask and optional Jupyter.
- Procfile — suitable for platforms like Render/Heroku with Gunicorn.

Add a deployment architecture diagram/screenshot later showing container/service, ports, and environment variables.

In [ ]:
# File: requirements.txt
Flask
joblib
nltk
google-generativeai
python-dotenv
gunicorn
scikit-learn

In [ ]:
# File: Procfile (as provided)
web:
gunicorn
app:app

# Note: A conventional Procfile is usually a single line: "web: gunicorn app:app"
# If using Render/Heroku, consider using the one-line format above.

In [ ]:
# File: DockerFile (note capital F in filename)
# Use official Python image
FROM python:3.11-slim

# Set environment variables
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV FLASK_APP=app.py
ENV FLASK_RUN_HOST=0.0.0.0

# Set the working directory in the container
WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \
    build-essential \
    curl \
    && rm -rf /var/lib/apt/lists/*

# Install Python dependencies
COPY requirements.txt .
RUN pip install --upgrade pip && pip install -r requirements.txt && pip install notebook

# Copy all source code to the container
COPY . .

# Expose the port your Flask app will run on
EXPOSE 5000

# Optional: expose Jupyter if you want to use notebooks inside the container
EXPOSE 8888

# Command to run both Flask and Jupyter (you can modify/remove Jupyter if unnecessary)
CMD ["sh", "-c", "flask run & jupyter notebook --ip=0.0.0.0 --port=8888 --allow-root --NotebookApp.token='' --NotebookApp.password=''"]

### 11.1 Environment Variables (.env)
Store secrets locally in `.env` and load via `python-dotenv` in development. Never commit secrets.

Add an image later showing environment variable configuration in your deployment platform.

In [ ]:
# File: .env (masking the actual key)
GOOGLE_API_KEY=********************************

## 12. Security, Ethics, and Privacy

- Secrets Management: Use environment variables or secret managers; never hardcode keys.
- Data Privacy: Avoid storing user-submitted texts unless consented and anonymized.
- Fairness: Monitor for bias; toxic language can correlate with dialects or reclaimed terms.
- Transparency: Communicate probabilistic nature and uncertainty in conclusions.
- Safety: Provide disclaimers and appeal mechanisms if deployed publicly.

## 13. Limitations and Future Work

- Domain Shift: Performance may drop on out-of-domain platforms or languages.
- Context Length: Short comments are harder to classify; consider context windows.
- Models: Explore transformer-based fine-tuning (DistilBERT/BERT) for improved recall.
- Explanations: Add SHAP/LIME explanations for per-prediction interpretability.
- Thresholding: Calibrate per-label thresholds using validation ROC/PR curves.
- UI/UX: Add mobile-responsive design and accessibility improvements.

## 14. Conclusion

CyberShield demonstrates a practical, efficient approach to cyberbullying detection using TF–IDF features and linear multi-label classification exposed via a Flask API. The system augments numeric outputs with generative explanations to improve user comprehension. With careful attention to bias, privacy, and deployment hygiene, this approach can be integrated into moderation workflows or educational tools.

## 15. References

- Jigsaw Toxic Comment Classification Challenge (Kaggle).
- Scikit-learn documentation for TF–IDF and classifiers.
- NLTK documentation.
- Google Generative AI Python SDK.
- Flask documentation.
- WordCloud library documentation.

## 16. Appendix: Full Code Listings and Artifacts

### 16.1 README.md (Project Overview in Repo)
The README provides a concise project description, features, tech stack, installation, usage, Docker instructions, and contributor info. Full text included below for completeness.

# CyberShield 🛡️🤖

Live On--- https://cybershield-q3je.onrender.com 


**Protecting Digital Spaces with AI – Stopping Cyberbullying**

## 🌟 Overview

CyberShield is an AI-powered tool designed to detect and prevent cyberbullying using **machine learning models**, **natural language processing (NLP)**, and a **Flask-based web service**.
By analyzing online conversations, CyberShield helps create a safer digital environment.

## ✨ Features

✔️ **AI-driven cyberbullying detection** using NLP models
✔️ **Flask-based web service** for API integration
✔️ **Interactive Web Interface** for analysis and insights
✔️ **Google Gemini AI-powered conclusions** for better contextual assessment

## 🛠️ Tech Stack

🔹 **Jupyter Notebook** – Model development & experimentation
🔹 **HTML** – Web interface for user interaction
🔹**Python 3.8+** – Core programming language
🔹 **Flask** – Web framework for API deployment
🔹 **ML Models (NLP-based)** – Detect toxic speech and cyberbullying

## 📦 Dependencies

CyberShield requires the following dependencies (found in `requirements.txt`):
🔹 Flask
🔹 joblib
🔹 nltk
🔹 google-generativeai
🔹 python-dotenv
🔹 gunicorn
🔹 scikit-learn

## ⚙️ Installation

### 🔽 Step 1: Clone the Repository

```sh
git clone <your-repo-url>
cd cybershield
```

### 🔽 Step 2: Set Up a Virtual Environment (Optional)

```
### Linux/macOS 🐧🍏
python3 -m venv venv
source venv/bin/activate

### Windows 🏁
python3 -m venv venv
.\venv\Scripts\activate
```

### 📌 Step 3: Install Dependencie

```
pip install -r requirements.txt
```

## 🚀 Usage

🔹 Start the Flask server to serve the AI-powered detection API.

```
python app.py
```

🔹 Submit comments for toxicity analysis using the /predict endpoint.
🔹 Generate AI-powered conclusions using the /conclude endpoint.
🔹 Integrate the API into existing platforms to enhance online safety.

## 🚀 Docker Installation & Setup
This project, CyberShield, can be containerized using Docker. Follow the steps below to build and run it.

### 📦 Build the Docker Image
Run the following command in the root of the project (where the Dockerfile is located):


docker build -t mpawank/cybershield:latest .

### ▶️ Run the Container

docker run -p 5000:5000 mpawank/cybershield:latest

Your Flask app will be available at: http://localhost:5000


## 👑 Project Owner

Built with 💡, 💻, and ❤️ by:

## [MpawanK](https://github.com/mpawank)

--🚀 Visionary behind CyberShield

--🔧 Developer • Designer • Defender of Digital Spaces

--📬 Connect: [GitHub](https://github.com/mpawank) | [LinkedIn](https://www.linkedin.com/in/m-pawan-kumar-b3978424b/)

"Building tools that protect, empower, and inspire safer online communities."

## 🤝 Contributing

We’re actively looking for contributors to help make the UI mobile responsive.
If you’re passionate about UX/UI, feel free to submit a pull request!

## ⭐ Star the repo and spread the word!

## 👥 Contributors

Thanks to all the amazing contributors helping make CyberShield better!<br> <br>
<a href="https://github.com/denshaw-09/contribute-Cybershield" target="_blank" rel="noopener noreferrer" style="text-decoration:none; text-align:center; display:inline-block;">
  <img src="https://github.com/denshaw-09.png" width="60" alt="denshaw-09 GitHub Avatar" style="border-radius:50%;" />
  <br />
  <sub>
    <b>denshaw-09</b>
  </sub>
</a>